## Data loads

In [ ]:
import os
from glob import glob
import random
from tqdm import tqdm
from datetime import datetime

# font_fname = os.environ.get('PROJECT_FONT_PATH', '')  # point at any local CJK/serif .ttf #적용할 폰트
import pandas as pd
import numpy as np

from matplotlib import pyplot as plt
from matplotlib import font_manager as fm
import seaborn as sbn

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM

from src.utils.preprocessing import Preprocessor
from src.models import SupervisedTextDataset, SupervisedPLModel, SimpleClassifier, F1MacroCallback

from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from umap import UMAP

import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint


font_fname = os.environ.get('PROJECT_FONT_PATH', '')  # point at any local CJK/serif .ttf #적용할 폰트
font_family = fm.FontProperties(fname=font_fname).get_name() #폰트 설정
plt.rcParams["font.family"] = font_family  #폰트 적용


In [ ]:
def save_dir_setup(model_name, prefix='CLF-recur', version='v001'):
    log_dir = os.path.join('..', '..', 'logs', datetime.now().date().isoformat(), f'{prefix}-{model_name}-{version}')
    os.makedirs(log_dir, exist_ok=True)

    figdir = os.path.join(log_dir,'figures')
    os.makedirs(figdir, exist_ok=True)

    filedir = os.path.join(log_dir,'files')
    os.makedirs(filedir, exist_ok=True)

    return dict(figure=figdir, file=filedir, log=log_dir)


def compute_alpha(df):
    # 클래스별 샘플 수 계산
    class_counts = df['target'].value_counts(sort=False).values
    class_freq = class_counts / class_counts.sum()
    alpha = np.exp(-class_freq / 1.)
    # alpha = (1.0 - class_freq) / (len(class_freq) - 1)
    alpha = alpha / alpha.sum()  # 정규화
    return alpha


def get_model_predictions(pl_model, data_loader):
    """
    Returns logits, targets, and predicted labels for a given model and dataloader.
    """
    pl_model.eval()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    pl_model.to(device)
    all_logits = []
    all_targets = []
    for batch in tqdm(data_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch.get('attention_mask', None)
        if attention_mask is not None:
            attention_mask = attention_mask.to(device)
        token_type_ids = batch.get('token_type_ids', None)
        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(device)
        labels = batch['labels'].to(device)
        with torch.no_grad():
            logits = pl_model(input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        all_logits.append(logits.cpu().numpy())
        all_targets.append(labels.cpu().numpy())
    all_logits = np.concatenate(all_logits)
    all_targets = np.concatenate(all_targets)
    all_preds = np.argmax(all_logits, axis=1)
    return dict(
        logits = all_logits, 
        targets = all_targets, 
        preds = all_preds
    )


def check_label(df, s):
    df = df.to_frame()
    df['label'] = s
    return df


def set_seed(seed):
    # 파이썬 해시 시드
    os.environ['PYTHONHASHSEED'] = str(seed)

    # 파이썬 random
    random.seed(seed)

    # numpy, torch, pytorch-lightning는 이미 다른 셀에서 import 되어 있으므로 그대로 사용
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # PyTorch Lightning utility (workers=True로 데이터로더 워커 시드도 설정)
    pl.seed_everything(seed, workers=True)



negative_patterns = ['without','not? (sugges|compat)', 'unlike', 'neither', 'no', 'negative']
uncertain_patterns = ['rule out', 'Rule Out', "Rule out"]

In [ ]:
raw_data_dir = glob(os.path.join('data', 'reports', '*'))
print(raw_data_dir)
df = pd.read_csv(raw_data_dir[0], encoding = 'utf-8-sig', engine = 'c').reset_index()

text_df = df['검사결과결론내용'].dropna()

preprocessor = Preprocessor(
    df = text_df,
    negative_patterns = negative_patterns,
    uncertain_patterns = uncertain_patterns
)


for gs, sdf in target_df.groupby(['병원등록번호','수술일자','수술실','진단명','수술명','검사접수일자','검사코드']):
    if sdf.shape[0] > 1:
        print(gs)
        print(sdf.shape)
    # break

### Rule based dataframe

In [ ]:
rule_path_dir = sorted(glob(os.path.join('data','rule_labels', '*labeled.csv')))
print(rule_path_dir)
rule_meta_path, rule_recur_path = rule_path_dir
rule_dict = dict(
    metas = pd.read_csv(rule_meta_path, encoding = 'utf-8-sig', engine = 'c'),
    recur = pd.read_csv(rule_recur_path, encoding = 'utf-8-sig', engine = 'c')
)


### 1st revision dataframe

In [ ]:
# reviewer gold workbooks
# metastasis_positive
revision_01_path_dir = sorted(glob(os.path.join('data','reviewer', '*positive.xlsx')))
print(revision_01_path_dir)
rev_01_meta_path, rev_01_recur_path = revision_01_path_dir
revision_dict = dict()
revision_dict[1] = dict(
    metas = pd.read_excel(rev_01_meta_path, engine = 'openpyxl').dropna(),
    recur = pd.read_excel(rev_01_recur_path, engine = 'openpyxl').dropna()
)


### 2nd revision dataframe

In [ ]:
revision_02_dir = os.path.join('data','reviewer')
revision_02_meta_path_dir = sorted(glob(os.path.join(revision_02_dir, '*meta*.xlsx')))
revision_02_recur_path_dir = sorted(glob(os.path.join(revision_02_dir, '*recur*.xlsx')))

print(revision_02_meta_path_dir)
print(revision_02_recur_path_dir)
revision_dict[2] = dict(
    metas = pd.concat([pd.read_excel(p, engine = 'openpyxl').dropna() for p in revision_02_meta_path_dir]),
    recur = pd.concat([pd.read_excel(p, engine = 'openpyxl').dropna() for p in revision_02_recur_path_dir])
)

## Duplicate Check

### Def function

In [ ]:
def agg_df_load(target):
    data_dict = preprocessor.target_filtering(target)
    merged_target_df = pd.concat([check_label(d, k) for k, d in data_dict.items()]).sort_index()
    neg_length = df.shape[0] - merged_target_df.shape[0]

    temp_df = df.copy()
    temp_df.loc[merged_target_df.index, f'{target}_text'] = merged_target_df['검사결과결론내용']
    temp_df.loc[merged_target_df.index, f'label'] = merged_target_df['label']
    target_df = temp_df.iloc[merged_target_df.index].copy()
    drop_df = temp_df.loc[~temp_df.index.isin(target_df.index)]

    df1 = target_df.reset_index()[['index','검사결과결론내용',target+'_text', 'label']].set_axis(['index','raw_text','prep_text','raw_label'], axis = 1)
    df2 = rule_dict[target][['index','검사결과결론내용','label']].set_axis(['index','rule_text','rule_label'], axis = 1)
    df3 = revision_dict[1][target].set_axis(['index','R1_text','R1_label'], axis = 1)
    df4 = revision_dict[2][target][['index','검사결과결론내용','재분류']].set_axis(['index','R2_text','R2_label'], axis = 1)

    print(df1.shape, df2.shape, df3.shape, df4.shape)

    agg_df = df1.merge(df2, on = 'index', how = 'outer')
    agg_df = agg_df.merge(df3, on = 'index', how = 'outer')
    agg_df = agg_df.merge(df4, on = 'index', how = 'outer')

    agg_df['n_samples'] = agg_df.groupby('prep_text')['prep_text'].transform('count')
    return dict(
        agg = agg_df, 
        drop = drop_df,
        total_length = len(df), 
        neg_length = neg_length, 
        R1 = df3,
        R2 = df4,
    )

    
def vis_confusion(cm, labels, save_strs, figsize = (7, 6), fm_size = 15):
    
    plt.rcParams['font.size'] = fm_size
    fig, ax = plt.subplots(1, 1, figsize=figsize)
    sbn.heatmap(
        cm, 
        annot=True, fmt='d', cmap='Blues',
        xticklabels=[i.capitalize() for i in labels],
        yticklabels=[i.capitalize() for i in labels], 
        ax=ax, 
        cbar = False
    )
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')
    fig.tight_layout()

    # target, rev_ver, pred_ver = save_strs 
    # fig.savefig(os.path.join(figdir, f'{target}_{rev_ver}_{pred_ver}.png'), dpi = 400)
    fig.savefig(os.path.join(figdir, '_'.join(save_strs) + '.png'), dpi = 400)
    plt.close()


def score2dict(yt, yp, n_samples = None):
    return dict(
        accuracy = accuracy_score(yt, yp, sample_weight = n_samples),
        f1 = f1_score(yt, yp, average = 'macro', sample_weight = n_samples),
        weighted_f1 = f1_score(yt, yp, average = 'weighted', sample_weight = n_samples),
    )

    

def sub_return(sub_df, dup_df, target_col, flag_dup = False):
    label_col = target_col + '_label'
    text_col = target_col + '_text'
    # if flag_dup:
    #     sub_df
    if flag_dup:
        dup_df = dup_df.drop_duplicates(text_col)
        sub_df = sub_df.drop_duplicates(text_col)
        sub_df = sub_df.loc[~sub_df.prep_text.isin(dup_df.prep_text)]


    sum_sub_df = pd.concat([
        sub_df[label_col].value_counts().loc[labels],
        dup_df[label_col].value_counts().loc[labels],
    ], axis = 1)
    sum_sub_df.loc[:, 'Sum'] = sum_sub_df.sum(1)
    sum_sub_df.loc['Sum'] = sum_sub_df.sum(0)
    
    return sub_df, dup_df

    
txt2label = {
    'meta' : 'positive',
    'recur' : 'positive',
    'positive' : 'positive',
    'uncertain' : 'uncertain',
}
textmapping_fn = lambda x: txt2label[x] if x in txt2label else 'negative'


### Recurrence

리커런스 중복 인덱스 샘플들이 실제로 몇개나 존재하는 샘플들인지 체크

Note: consolidating the multiple human-review rounds into the reviewed labels is a study-design step described in the paper; here we treat the consolidated reviewed labels as a given input. The preprocessing -> embedding -> clustering weak-negative extraction below is fully reproducible.

In [ ]:
dup_flag = True
target = 'metas'
labels = ['positive','uncertain','negative'] if 'recur' == target else ['positive','uncertain','regional','negative']

agg_dict = agg_df_load(target)
agg_df = agg_dict.pop('agg')
drop_df = agg_dict.pop('drop')
r1_df = agg_dict.pop('R1')
r2_df = agg_dict.pop('R2')


In [ ]:
cover_index = agg_df.R1_label.notna() | agg_df.R2_label.notna()
dup_index = agg_df.R1_label.notna() & agg_df.R2_label.notna()
error_index = (agg_df.R1_label != agg_df.R2_label) & dup_index
cover_index = cover_index & ~error_index
dup_index = dup_index & ~error_index

cover_df = agg_df.loc[cover_index].drop_duplicates('prep_text')


In [ ]:
un_df = agg_df.loc[~agg_df['index'].isin(cover_df['index']) & ~agg_df['prep_text'].isin(cover_df['prep_text'])].drop_duplicates('prep_text')

In [ ]:
gt_df = cover_df.copy()
gt_df.loc[:, 'R_text'] = gt_df['R1_text'].combine_first(gt_df['R2_text'])
gt_df.loc[:, 'label'] = gt_df['R1_label'].combine_first(gt_df['R2_label']).apply(textmapping_fn)
print((gt_df.R_text == gt_df.prep_text).mean())

### Filtering

In [ ]:
import numpy as np

from matplotlib import pyplot as plt
import seaborn as sbn

from sklearn.ensemble import RandomForestClassifier
from sklearn.manifold import Isomap, TSNE
from sklearn.cluster import DBSCAN
from umap import UMAP
from sklearn.metrics import confusion_matrix, classification_report

from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import torch

from transformers import AutoModel, AutoTokenizer


class TextDataset(Dataset):
    def __init__(self, text_series):
        self.text_series = text_series.tolist()

    def __len__(self):
        return len(self.text_series)

    def __getitem__(self, idx):
        return self.text_series[idx]
        
        

@torch.no_grad()
def llm_embedding_fn(text_df):

    language_model.eval()
    language_model.cuda()

    dl = DataLoader(
        TextDataset(text_df),
        batch_size = 128,
        shuffle = False
    )

    vector_outputs = []
    # ids_outputs = []
    for text in tqdm(dl):
        token = tokenizer(
            text,
            # [tokenizer.special_tokens_map['cls_token']+t+tokenizer.special_tokens_map['sep_token'] for t in text], 
            return_tensors='pt', padding=True, truncation=True, padding_side='right'
            )
        # input_ids = token['input_ids']
        # with torch.no_grad():
        output = language_model(**{k:v.cuda() for k , v in token.items()}).last_hidden_state[:,0].cpu().float().numpy()
        
        vector_outputs.append(output)
    # ids_outputs.append(input_ids.numpy())

    return vector_outputs



model_path = os.path.join('..','..','..','..','model','PubMedBERT-base-uncased-sts-combined')

tokenizer = AutoTokenizer.from_pretrained(model_path)
language_model = AutoModel.from_pretrained(
        model_path,
        dtype="bfloat16",
    )

txt2label = {
    'meta' : 'positive',
    'recur' : 'positive',
    'positive' : 'positive',
    'uncertain' : 'uncertain',
}
textmapping_fn = lambda x: txt2label[x] if x in txt2label else 'negative'


In [ ]:
gt_vector_outputs = llm_embedding_fn(gt_df['prep_text'])
un_vector_outputs = llm_embedding_fn(un_df['prep_text'])

gt_outputs = np.concatenate([i for i in gt_vector_outputs], axis = 0)
un_outputs = np.concatenate([i for i in un_vector_outputs], axis = 0)

In [ ]:
gt_df

In [ ]:
manifold = UMAP()
manifold_eos_vectors = manifold.fit_transform(un_outputs)
gt_manifold_eos_vector = manifold.transform(gt_outputs)

In [ ]:
# db = DBSCAN(eps = 1) # recur
db = DBSCAN(eps = 1e-1) # metas
numb_cluster = db.fit_predict(manifold_eos_vectors)


In [ ]:
# sub_manifold_vectors = manifold_eos_vectors[:part_df.shape[0], :]
# sub_numb_cluster = numb_cluster[:part_df.shape[0]]

sub_manifold_vectors = manifold_eos_vectors
sub_numb_cluster = numb_cluster

plot_df = pd.DataFrame(
    np.concatenate([sub_manifold_vectors, sub_numb_cluster[:, None]], axis = 1),
    columns = ['x1','x2','c']
)

sbn.jointplot(
    data = plot_df, x = 'x1', y = 'x2', 
    hue = 'c',
    joint_kws = {'s' : 2, 'alpha' : .5},
    )

In [ ]:
from collections import Counter
counter = Counter(numb_cluster)
most_common = counter.most_common()[0][0]
np.where(numb_cluster != most_common)[0]

In [ ]:
neg_texts = []
test_texts = []
cumsum = 0

train_label_counts = gt_df.label.value_counts()
pos_thr = train_label_counts['positive'] - train_label_counts['negative']
for c, _ in counter.most_common():
    sub_df = un_df.iloc[np.where(numb_cluster == c)]
    unique_label = sub_df['rule_label'].unique()
    if ('negative' in unique_label) and (len(unique_label) == 1) and (cumsum < pos_thr):
        neg_texts.append(sub_df)
        cumsum += sub_df.shape[0]
    else:
        test_texts.append(sub_df)

un_agg_df = pd.concat(test_texts)

neg_df = pd.concat(neg_texts)
neg_df.loc[:, 'label'] = 'negative'
print(neg_df.prep_text.value_counts().unique(), neg_df.label.value_counts().item())
# gt_agg_df = pd.concat([gt_df, neg_df])
neg_df

In [ ]:
test_df = un_agg_df.copy()
test_df = test_df.drop_duplicates('prep_text')
train_df = pd.concat([gt_df, neg_df.loc[~neg_df.prep_text.isin(test_df.prep_text)]])

In [ ]:
set(train_df['index'].tolist()).intersection(test_df['index'].tolist()), set(test_df['index'].tolist()).intersection(train_df['index'].tolist())

In [ ]:
set(train_df['prep_text'].tolist()).intersection(test_df['prep_text'].tolist()), set(test_df['prep_text'].tolist()).intersection(train_df['prep_text'].tolist())

In [ ]:
pd.concat([train_df, test_df]).shape

In [ ]:
train_df.isna().sum()

In [ ]:
test_df.isna().sum()

In [ ]:
test_df

In [ ]:
from datetime import datetime
today = datetime.now().date().isoformat()
log_save_dir = os.path.join('outputs','splits', 'all')
os.makedirs(log_save_dir, exist_ok=True)
train_df.to_excel(os.path.join(log_save_dir, f'{target}_train_df.xlsx'), index = False)
test_df.to_excel(os.path.join(log_save_dir, f'{target}_test_df.xlsx'), index = False)